# Southern California Seismic Station Map — 31 October 1991

**Time window:** 06:00–09:00 PST = 14:00–17:00 UTC (DST ended 27 Oct 1991; PST = UTC−8 was in effect)  
**Coverage:** All operating broadband (BH\*, HH\*, LH\*) and strong-motion (HN\*, BN\*, EN\*) channels  
**Special focus:** Pinyon Flat Observatory (PFO) and the II (IRIS/IDA) + AZ (ANZA/SIO) networks co-located there  

Data fetched from IRIS FDSN. Internet connection required.

In [ ]:
from obspy import UTCDateTime
from obspy.clients.fdsn import Client
from obspy.clients.fdsn.header import FDSNNoDataException
import pygmt
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print('PyGMT version:', pygmt.__version__)

In [ ]:
# ── Time window ──────────────────────────────────────────────────────────────
# 31 Oct 1991 06:00–09:00 PST  →  14:00–17:00 UTC
# (Daylight Saving ended Sun 27 Oct 1991; PST = UTC−8 was in effect by the 31st)
t_start = UTCDateTime('1991-10-31T14:00:00')
t_end   = UTCDateTime('1991-10-31T17:00:00')

# ── Southern California bounding box ─────────────────────────────────────────
MINLAT, MAXLAT =  32.0,  36.5
MINLON, MAXLON = -121.0, -114.0
REGION = [MINLON, MAXLON, MINLAT, MAXLAT]

# ── Pinyon Flat Observatory reference location ────────────────────────────────
PFO_LON, PFO_LAT = -116.4555, 33.6106   # UCSD / SIO

# ── Channel patterns ─────────────────────────────────────────────────────────
# Broadband: BH? HH? LH?  |  Strong-motion accelerometers: HN? BN? EN?
CHANNELS = 'BH?,HH?,LH?,HN?,BN?,EN?'

print(f'Query window : {t_start}  →  {t_end}')
print(f'Region       : lon [{MINLON}, {MAXLON}]  lat [{MINLAT}, {MAXLAT}]')
print(f'Channels     : {CHANNELS}')

In [ ]:
# ── Query IRIS FDSN for station inventory ─────────────────────────────────────
# We query at channel level so we can distinguish BB vs SM instruments.
# Networks active in S. California in 1991:
#   CI  – Southern California Seismic Network (Caltech / USGS)
#   TI  – TERRASCOPE broadband network (Caltech)          → became part of CI later
#   II  – IRIS/IDA global broadband (includes PFO)
#   AZ  – ANZA network (SIO / UCSD, includes stations near PFO)
#   CE  – California Strong Motion Instrumentation Program (CSMIP)
#   NP  – USGS National Strong Motion Program

client = Client('IRIS')

print('Querying IRIS FDSN (channel-level) …')
try:
    inv = client.get_stations(
        network='*',
        channel=CHANNELS,
        starttime=t_start,
        endtime=t_end,
        minlatitude=MINLAT,
        maxlatitude=MAXLAT,
        minlongitude=MINLON,
        maxlongitude=MAXLON,
        level='channel',
    )
    print(inv)
except FDSNNoDataException:
    print('No data returned – check network connectivity or time window.')
    inv = None

In [ ]:
# ── Parse inventory into a tidy DataFrame ─────────────────────────────────────
BB_PREFIXES = {'BH', 'HH', 'LH'}
SM_PREFIXES = {'HN', 'BN', 'EN'}

rows = {}

if inv is not None:
    for net in inv:
        for sta in net:
            key = (net.code, sta.code)
            if key not in rows:
                rows[key] = {
                    'network'   : net.code,
                    'station'   : sta.code,
                    'latitude'  : sta.latitude,
                    'longitude' : sta.longitude,
                    'elevation' : sta.elevation,
                    'has_bb'    : False,
                    'has_sm'    : False,
                    'channels'  : [],
                }
            for cha in sta:
                rows[key]['channels'].append(cha.code)
                if cha.code[:2] in BB_PREFIXES:
                    rows[key]['has_bb'] = True
                if cha.code[:2] in SM_PREFIXES:
                    rows[key]['has_sm'] = True

df = pd.DataFrame(rows.values())

if df.empty:
    print('WARNING: no stations found. Proceeding with empty dataset.')
else:
    def inst_type(row):
        if row.has_bb and row.has_sm: return 'Both'
        if row.has_bb:                return 'Broadband'
        return 'Strong Motion'

    df['type'] = df.apply(inst_type, axis=1)
    print(f'Total unique stations: {len(df)}')
    print(df.groupby(['network', 'type'])['station'].count().to_string())

In [ ]:
# ── Pinyon Flat / ANZA network detail ─────────────────────────────────────────
# PFO is at ~33.61°N 116.46°W; ANZA stations cluster ~30–50 km radius.
# Networks to highlight: II (IRIS/IDA), AZ (ANZA/SIO)

PFO_NETWORKS = {'II', 'AZ', 'TI'}

if not df.empty:
    pfo_area = df[
        (df.latitude  .between(32.8, 34.2)) &
        (df.longitude .between(-117.2, -115.5))
    ].copy()

    print(f'Stations near Pinyon Flat area ({len(pfo_area)}):' )
    print(pfo_area[['network', 'station', 'latitude', 'longitude', 'type', 'channels']].to_string(index=False))

In [ ]:
# ── Build PyGMT relief map ─────────────────────────────────────────────────────
import matplotlib
matplotlib.use('Agg')

fig = pygmt.Figure()

# ── Topographic relief (SRTM/ETOPO1, 1-arc-minute) ───────────────────────────
print('Downloading/caching earth relief grid …')
grid = pygmt.datasets.load_earth_relief(
    resolution='01m',
    region=REGION,
    registration='gridline',
)

# Colour palette: topo / geo with hill-shading
pygmt.makecpt(cmap='geo', series=[-3000, 4000, 100], continuous=True)

fig.grdimage(
    grid=grid,
    projection='M18c',
    region=REGION,
    cmap=True,
    shading='+a315+nt0.8',   # hill-shade: azimuth 315°, intensity 0.8
    frame=False,
)

# ── Coastlines, political borders, rivers ────────────────────────────────────
fig.coast(
    region=REGION,
    projection='M18c',
    shorelines='1/0.6p,black',
    borders=['1/0.7p,gray20', '2/0.4p,gray50'],
    water='lightcyan@30',
    rivers='a/0.4p,royalblue@40',
    frame=['WSne+t"S. California Seismic Stations | 31 Oct 1991 | 06–09 PST"',
           'xa2f1g2', 'ya1f0.5g1'],
)

# ── Station symbols ───────────────────────────────────────────────────────────
SYMBOL_SIZE = '0.28c'

if not df.empty:
    bb   = df[df.type == 'Broadband']
    sm   = df[df.type == 'Strong Motion']
    both = df[df.type == 'Both']

    if len(sm):
        fig.plot(
            x=sm.longitude.values, y=sm.latitude.values,
            style=f't{SYMBOL_SIZE}',   # triangle
            fill='orangered', pen='0.5p,black',
            label=f'Strong Motion ({len(sm)})',
        )
    if len(bb):
        fig.plot(
            x=bb.longitude.values, y=bb.latitude.values,
            style=f'i{SYMBOL_SIZE}',   # inverted triangle
            fill='dodgerblue', pen='0.5p,black',
            label=f'Broadband ({len(bb)})',
        )
    if len(both):
        fig.plot(
            x=both.longitude.values, y=both.latitude.values,
            style=f's{SYMBOL_SIZE}',   # square
            fill='gold', pen='0.5p,black',
            label=f'BB + SM ({len(both)})',
        )

    # ── Highlight II / AZ / TI network stations ───────────────────────────────
    pfo_nets = df[df.network.isin(PFO_NETWORKS)]
    if len(pfo_nets):
        fig.plot(
            x=pfo_nets.longitude.values, y=pfo_nets.latitude.values,
            style='c0.38c',   # circle outline
            fill='none', pen='1.5p,magenta',
            label=f'II / AZ / TI networks ({len(pfo_nets)})',
        )

# ── Pinyon Flat Observatory reference marker ──────────────────────────────────
fig.plot(
    x=[PFO_LON], y=[PFO_LAT],
    style='a0.55c',   # star
    fill='magenta', pen='1p,black',
    label='Pinyon Flat Obs. (PFO)',
)
fig.text(
    x=PFO_LON + 0.15, y=PFO_LAT + 0.08,
    text='PFO',
    font='7p,Helvetica-Bold,magenta',
    justify='LM',
    no_clip=True,
)

# ── Colourbar, legend, scale ──────────────────────────────────────────────────
fig.colorbar(
    frame=['a1000f500', 'x+lElevation (m)'],
    position='JBC+w10c+o0c/-1.2c+h',
)
fig.legend(position='JBR+jBR+o0.3c/1.5c', box='+gwhite@20+p0.5p')
fig.basemap(map_scale='g-118/32.4+w100k+f+lkm')

# ── Save ──────────────────────────────────────────────────────────────────────
out_path = '/workspace/data/socal_stations_1991.png'
fig.savefig(out_path, dpi=200)
print(f'Map saved → {out_path}')
fig.show()

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────
if not df.empty:
    summary = (
        df.groupby(['network', 'type'])['station']
          .count()
          .reset_index()
          .rename(columns={'station': 'count'})
          .sort_values(['network', 'type'])
    )
    print(summary.to_string(index=False))
    print(f'\nTotal: {len(df)} stations across {df.network.nunique()} networks')